In [1]:
%pip install -qU langchain-community pymupdf
!pip install -qU langchain-huggingface sentence-transformers
!pip install -qU langchain-groq
!pip install faiss-cpu

# 1. Loading the document

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "/content/the-quran-with-annotated-interpretation-in-modern-english-ali-unal.pdf"
loader = PyMuPDFLoader(file_path)

In [3]:
docs = loader.load()
# skiping empty pages
non_empty_docs = [d for d in docs if d.page_content.strip()]

# 2. Spliting document into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10000, # 10000 charecters long text
    chunk_overlap=200, # 200 charecters long overlapping
)

split_docs = text_splitter.split_documents(non_empty_docs)

# 3. Embeddings Model

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [6]:
texts = [doc.page_content for doc in split_docs] # convert documents into list[str]

In [7]:
split_docs_embeddings = embed_model.embed_documents(texts) # generate embeddings (list[list[float]])

# 4. FAISS (Facebook AI Similarity Search) vector database

In [8]:
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(
    documents=split_docs,
    embedding=embed_model,
)

# 5. LLM. GROQ (llama-3.3-70b-versatile)

In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="your_api_key"
)

# 6. Building Prompts and chains

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

In [11]:
def ask(query):
    prompt_template = PromptTemplate.from_template(
        "Give answer according to the following passages in the quran:\n{context}\n\n"
        "If the answer is not present in the given context then give answer according to "
        "the internet sources, but do inform that there are no passages in the quran "
        "about the question.\n\n"
        "Answer the following question:\n{question}"
    )

    def get_joined_context(q):
        docs = faiss_db.similarity_search(q, k=10)
        return "\n\n".join(d.page_content for d in docs)

    parallel_chain = RunnableParallel(
        context=RunnableLambda(get_joined_context),
        question=RunnablePassthrough()
    )

    chain = parallel_chain | prompt_template | llm | StrOutputParser()

    return chain.invoke(query)

Relevant questions

In [12]:
answer = ask("What does the Quran say about Day of Judgment?")
print(answer)

The Quran mentions the Day of Judgment in several passages. Here are some of the key points:

1. **The Day of Judgment is a day of reckoning**: The Quran states that on this day, every soul will be held accountable for its deeds (Surah 82:19, Surah 21:47).
2. **It is a day of absolute justice**: The Quran emphasizes that on the Day of Judgment, God will set up balances of absolute justice, and no person will be wronged in the least (Surah 21:47).
3. **Every deed will be weighed**: The Quran states that every deed, no matter how small, will be weighed on the Day of Judgment (Surah 21:47).
4. **The Day of Judgment is a day of separation**: The Quran mentions that on this day, people will be separated from one another, and the believers will be distinguished from the disbelievers (Surah 30:14).
5. **The disbelievers will be punished**: The Quran states that the disbelievers will be punished in the Blazing Flame (Surah 82:14-16, Surah 56:93-94).
6. **The believers will be rewarded**: The Q

Irrelevant questions

In [14]:
answer = ask("When did dinosaurs came into being?")
print(answer)

There is no passage in the Quran that specifically mentions dinosaurs or their origin. The Quran does mention the creation of animals and living beings, but it does not provide a detailed account of the history of life on Earth or the emergence of specific species like dinosaurs.

According to internet sources, dinosaurs are believed to have originated during the Middle to Late Triassic period, around 230-245 million years ago. They dominated Earth's landscapes during the Mesozoic Era, which lasted until about 65 million years ago, when they became extinct.

Please note that the answer is based on scientific knowledge and not on any specific passage from the Quran, as the Quran does not provide information on this topic.
